In [62]:
import pickle
import pandas as pd
import itertools
from collections import Counter
import numpy as np
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from gensim.models import word2vec
from sklearn.linear_model import LogisticRegression
import os
import string


# NLTK library downloads
nltk.download('stopwords')
nltk.download('punkt') 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alexh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alexh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [63]:
df_train = pd.read_csv("train.csv")

In [64]:
df_test = pd.read_csv("test.csv")

In [65]:
def create_features(df):
    df["attributes.OutdoorSeating"] = df_train["attributes.OutdoorSeating"].apply(lambda x: 1 if x == b'True' else 0)
    df["attributes.RestaurantsDelivery"] = df_train["attributes.RestaurantsDelivery"].apply(lambda x: 1 if x == b'True' else 0)
    df["attributes.RestaurantsReservations"] = df_train["attributes.RestaurantsReservations"].apply(lambda x: 1 if x == b'True' else 0)
    df["attributes.GoodForKids"] = df_train["attributes.GoodForKids"].apply(lambda x: 1 if x == b'True' else 0)
    df["attributes.RestaurantsTakeOut"] = df_train["attributes.RestaurantsTakeOut"].apply(lambda x: 1 if x == b'True' else 0)
    df["attributes.RestaurantsGoodForGroups"] = df_train["attributes.RestaurantsGoodForGroups"].apply(lambda x: 1 if x == b'True' else 0)
    return df[["attributes.OutdoorSeating", "attributes.RestaurantsDelivery", "attributes.RestaurantsReservations", "attributes.GoodForKids", "attributes.RestaurantsTakeOut", "attributes.RestaurantsGoodForGroups"]]

In [66]:
df_train2 = create_features(df_train)
df_test2 = create_features(df_test)

In [67]:
from nltk.corpus import stopwords 
stop = set(stopwords.words('english'))

from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer

In [68]:
ps = PorterStemmer()

def pre_processing_by_nltk(doc,stemming=True, need_sent=False):
    sentences = sent_tokenize(doc)
    tokens = []
    for sent in sentences:
        words = word_tokenize(sent)
        if stemming:
            words = [ps.stem(word) for word in words]
        if need_sent:
            tokens.append(words)
        else:
            tokens += words
    return [w.lower() for w in tokens if w.lower() not in stop]

In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [70]:
tfidf = TfidfVectorizer(strip_accents=None,
                       lowercase=True,
                       preprocessor=None,
                       tokenizer = pre_processing_by_nltk,
                        ngram_range=(1, 3),
                        min_df = 5,
                       use_idf = True,
                       norm = 'l2',
                       smooth_idf = True)

In [71]:
train_review_tfidf = tfidf.fit_transform(df_train["review"]) 

C:\Users\alexh\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [72]:
test_review_tfidf = tfidf.transform(df_test["review"])

In [73]:
from scipy.sparse import hstack

In [74]:
X_train_tfidf = hstack([
    train_review_tfidf,
    df_train2[[
        "attributes.OutdoorSeating",
        "attributes.RestaurantsDelivery",
        "attributes.RestaurantsReservations",
        "attributes.GoodForKids",
        "attributes.RestaurantsTakeOut",
        "attributes.RestaurantsGoodForGroups"
    ]].values
])

In [75]:
X_test_tfidf = hstack([
    test_review_tfidf,
    df_test2[[
        "attributes.OutdoorSeating",
        "attributes.RestaurantsDelivery",
        "attributes.RestaurantsReservations",
        "attributes.GoodForKids",
        "attributes.RestaurantsTakeOut",
        "attributes.RestaurantsGoodForGroups"
    ]].values
])

In [76]:
Y_train = df_train["label"]

In [78]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=False)
X_train_idf = scaler.fit_transform(X_train_tfidf)
X_test_idf = scaler.transform(X_test_tfidf)

In [79]:
from sklearn.linear_model import LogisticRegression

In [80]:
classifier = LogisticRegression(class_weight='balanced')

In [81]:
from sklearn.model_selection import GridSearchCV

In [82]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],      
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'newton-cg', 'lbfgs']
}

In [83]:
grid = GridSearchCV(classifier, param_grid, cv=5, scoring='accuracy', verbose=1)
grid.fit(X_train_tfidf, Y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


C:\Users\alexh\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\alexh\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

GridSearchCV(cv=5, estimator=LogisticRegression(class_weight='balanced'),
             param_grid={'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear', 'newton-cg', 'lbfgs']},
             scoring='accuracy', verbose=1)

In [84]:
preds = grid.predict(X_test_tfidf)

In [85]:
# in your implemetation, create the output file using the same format
dic = {"Id": [], "Predicted": []}
for i, pred in enumerate(preds):
    dic["Id"].append(i)
    dic["Predicted"].append(pred)

dic_df = pd.DataFrame.from_dict(dic)
dic_df.to_csv("predicted.csv", index=False)